# sahformer — scaled base training

Resumable. Kaggle kills the session before 400k steps finish, so **re-run this notebook** and it
continues from the last checkpoint instead of restarting.

**Each session:**
1. Run all → it trains until Kaggle stops it, checkpointing every 2000 steps
2. *Save Version* (commit) so `/kaggle/working` is preserved as an output
3. Next session: add that output as an input dataset, then run again — it resumes

**What this trains:** backbone scaled to ~30M (dim 512, 12 blocks) plus an Elo head.
**Not** the bucket time head — the 2026-09-03 findings showed timing is fixed at inference
(sample rather than average; self-clock), not by retraining.

**Data caveat:** the HF corpus is `elo_self` 2012–3074. This answers *does capacity improve
move-match*, not *does the base serve clonefish users*, who sit below 2000.

In [ ]:
!pip install -q python-chess zstandard huggingface_hub

In [ ]:
# ---- CONFIG: edit these two lines ----------------------------------------
BUNDLE = "/kaggle/input/sahformer-code"          # dataset holding sahformer/ + scripts/
SHARDS = ""    # leave "" to download from HF, or point at a mounted shards dataset
PREV    = []   # e.g. ["/kaggle/input/sahformer-base-run/ckpt"] - previous session's output
# --------------------------------------------------------------------------
import sys, os, glob
sys.path.insert(0, BUNDLE)
os.makedirs("/kaggle/working/ckpt", exist_ok=True)
print("code:", sorted(os.listdir(BUNDLE))[:6])

In [ ]:
# Shards: from a mounted dataset if given, else pull from HF.
# HF token goes in Kaggle Secrets as HF_TOKEN (Add-ons -> Secrets) - the dataset is private.
if not SHARDS:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import snapshot_download
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    SHARDS = snapshot_download(repo_id="slobaspeed/chesscom-balanced-shards",
                               repo_type="dataset", token=tok,
                               local_dir="/kaggle/working/shards")
print(SHARDS, "->", len(glob.glob(os.path.join(SHARDS, '**', '*.npz'), recursive=True)), "shards")

In [ ]:
import subprocess, sys
cmd = [sys.executable, os.path.join(BUNDLE, "scripts", "kaggle_train.py"),
       "--shards", SHARDS,
       "--out", "/kaggle/working/ckpt",
       "--dim", "512", "--blocks", "12", "--heads", "8",
       "--bs", "256", "--lr", "4e-5", "--steps", "400000",
       "--save-every", "2000", "--log-every", "200"]
if PREV:
    cmd += ["--resume-dirs"] + PREV
env = dict(os.environ, PYTHONPATH=BUNDLE)
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()

In [ ]:
# progress so far (safe to run even if training was interrupted)
import json
h = json.load(open("/kaggle/working/ckpt/history.json"))
print(f"{len(h)} logged points, up to step {h[-1]['step']:,}")
for r in h[::max(len(h)//12,1)]:
    print(f"  step {r['step']:>7} loss {r['loss']:.4f} move_acc {r['move_acc']*100:.2f}%")